# Map of all compounds in the input dataset

In [71]:
INPUT_FILE = "Compounds.xlsx"
INPUT_DIRECTORY = "../data/"
OUTPUT_DIRECTORY = "../results/"

In [109]:
import pandas as pd
import os
import requests
import json
import nistchempy as nist

In [ ]:
df = pd.read_excel(os.path.join(INPUT_DIRECTORY, INPUT_FILE), header=None)
df.head()

df.rename(columns={0: "Input Name"}, inplace=True)

df.insert(1, "Common/IUPAC Name", None)
df.insert(2, "InChIKey", None)
df.insert(3, "CID", None)
df.insert(4, "RI Polar (NIST)", None)
df.insert(5, "HMDB ID", None)
df.insert(6, "Ontology Links", None)
df.insert(7, "Pathway", None)
df.insert(8, "Disease Association", None)
df.insert(2, "InChI", None)
df.insert(10, "SMILES", None)

df.head(5)

## PubChem API

In [ ]:
PUBCHEM_API_URL = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound"

compound_synonyms = pd.DataFrame()
compound_synonyms.insert(0, "Input Name", None)
compound_synonyms.insert(1, "CID", None)

for index, row in df.iterrows():
    compound_name = row["Input Name"]
    pubchem_synonyms_result = requests.get(f"{PUBCHEM_API_URL}/name/{compound_name}/synonyms/JSON").json()
    if "InformationList" not in pubchem_synonyms_result or len(pubchem_synonyms_result["InformationList"]["Information"]) == 0:
        print(f"No result in PubChem found for {index}: {compound_name}")
        continue
    synonyms = pubchem_synonyms_result["InformationList"]["Information"][0]["Synonym"]
    cid = pubchem_synonyms_result["InformationList"]["Information"][0]["CID"]
    compound_synonyms.at[index, "Input Name"] = compound_name
    compound_synonyms.at[index, "CID"] = cid
    df.at[index, "CID"] = cid
    for i, s in enumerate(synonyms):
        if i > 20: # limit to 20 synonyms to avoid too many columns
            break
        if i >= len(compound_synonyms.columns) - 2: # if we need more columns, add them
            compound_synonyms.insert(len(compound_synonyms.columns), f"Synonym {i}", None)
        compound_synonyms.at[index, f"Synonym {i}"] = s

No result in PubChem found for 10: 1,2-Benzenedicarboxylic acid, bis(2-methylpropyl) ester, 
No result in PubChem found for 43: 1H-Inaene, 2,3-dihydro-2-methyl-
No result in PubChem found for 57: 2-Buten-1-ol, 2-ethyl-4-(2,2,3-trimethyl-3-cyclopenten-1-yl)-
No result in PubChem found for 99: 4,7-Methano-1H-inaene, octahydro-2-methylene-
No result in PubChem found for 101: 4,7-methano-1H-inaen-5-ol, 3a,4,5,6,7,7a-hexahydro-, acetate
No result in PubChem found for 149: Benzene, (1-ethylunaecyl)-
No result in PubChem found for 156: Benzene, (1-methylunaecyl)-
No result in PubChem found for 199: Carbonic acid, decyl unaecyl ester
No result in PubChem found for 209: Cyclobuta[1,2:3,4]dicyclooctene, 1,2,5,6,6a,6b,7,8,11,12,12a,12b-dodecahydro-, (6a.alpha.,6b.beta.,12a.alpha.,12b.beta.)-
No result in PubChem found for 214: Cyclohexanol, 5-methyl-2-(1-methylethyl)-, (1.alpha.,2.beta.,5.alpha.)-(.+/-.)-
No result in PubChem found for 215: Cyclohexanol, 5-methyl-2-(1-methylethyl)-, (1α,2β,5α)-
N

In [103]:
for index, row in df.iterrows():
    cid = row["CID"]
    pubchem_synonyms_result = requests.get(f"{PUBCHEM_API_URL}/cid/{cid}/property/Title,SMILES,InChI,InChIKey,IUPACName/JSON").json()
    if "PropertyTable" not in pubchem_synonyms_result or len(pubchem_synonyms_result["PropertyTable"]["Properties"]) == 0:
        print(f"No result in PubChem found for CID {cid}")
        continue
    props = pubchem_synonyms_result["PropertyTable"]["Properties"][0]
    df.at[index, "Common/IUPAC Name"] = props["IUPACName"]
    df.at[index, "InChI"] = props["InChI"]
    df.at[index, "InChIKey"] = props["InChIKey"]
    df.at[index, "SMILES"] = props["SMILES"]

df.head(5)

No result in PubChem found for CID nan
No result in PubChem found for CID nan
No result in PubChem found for CID nan
No result in PubChem found for CID nan
No result in PubChem found for CID nan
No result in PubChem found for CID nan
No result in PubChem found for CID nan
No result in PubChem found for CID nan
No result in PubChem found for CID nan
No result in PubChem found for CID nan
No result in PubChem found for CID nan
No result in PubChem found for CID nan
No result in PubChem found for CID nan
No result in PubChem found for CID nan
No result in PubChem found for CID nan


,Input Name,Common/IUPAC Name,InChI,InChIKey,CID,RI Polar (NIST),HMDB ID,Ontology Links,Pathway,Disease Association,SMILES
0,(+)-Isomenthol,"(1S,2R,5R)-5-methyl-2-propan-2-ylcyclohexan-1-ol",InChI=1S/C10H20O/c1-7(2)9-5-4-8(3)6-10(9)11/h7...,NOOLISFMXDJSKH-BBBLOLIVSA-N,45056,None,None,None,None,None,C[C@@H]1CC[C@@H]([C@H](C1)O)C(C)C
1,"(1R)-2,6,6-Trimethylbicyclo[3.1.1]hept-2-ene","(1R,5R)-2,6,6-trimethylbicyclo[3.1.1]hept-2-ene","InChI=1S/C10H16/c1-7-4-5-8-6-9(7)10(8,2)3/h4,8...",GRWFGVWFFZKLTI-RKDXNWHRSA-N,82227,None,None,None,None,None,CC1=CC[C@@H]2C[C@H]1C2(C)C
2,(E)-Hexadec-2-enal,(E)-hexadec-2-enal,InChI=1S/C16H30O/c1-2-3-4-5-6-7-8-9-10-11-12-1...,KLJFYXOVGVXZKT-CCEZHUSRSA-N,5280541,None,None,None,None,None,CCCCCCCCCCCCC/C=C/C=O
3,.alpha. Isomethyl ionone,"(E)-3-methyl-4-(2,6,6-trimethylcyclohex-2-en-1...","InChI=1S/C14H22O/c1-10-7-6-8-14(4,5)13(10)9-11...",JRJBVWJSTHECJK-PKNBQFBNSA-N,5372174,None,None,None,None,None,CC1=CCCC(C1/C=C(\C)/C(=O)C)(C)C
4,.alpha.-Methyl-.alpha.-[4-methyl-3-pentenyl]ox...,6-methyl-2-(oxiran-2-yl)hept-5-en-2-ol,"InChI=1S/C10H18O2/c1-8(2)5-4-6-10(3,11)9-7-12-...",BXOURKNXQXLKRK-UHFFFAOYSA-N,102611,None,None,None,None,None,CC(=CCCC(C)(C1CO1)O)C


In [73]:
compound_synonyms.to_csv(os.path.join(OUTPUT_DIRECTORY, "compound_synonyms.csv"), index=False)
compound_synonyms.head()

,Input Name,CID,Synonym 0,Synonym 1,Synonym 2,Synonym 3,Synonym 4,Synonym 5,Synonym 6,Synonym 7,...,Synonym 11,Synonym 12,Synonym 13,Synonym 14,Synonym 15,Synonym 16,Synonym 17,Synonym 18,Synonym 19,Synonym 20
0,(+)-Isomenthol,45056,(+)-Isomenthol,23283-97-8,D-Isomenthol,"Isomenthol, (+)-","Cyclohexanol, 5-methyl-2-(1-methylethyl)-, (1S...",RCZ2YJZ08S,"(1S,2R,5R)-(+)-Isomenthol","ISOMENTHOL, D-",...,"MENTHOL, (1R,3S,4R)-(+)-","Cyclohexanol, 5-methyl-2-(1-methylethyl)-, (1S...","CYCLOHEXANOL, 5-METHYL-2-(1-METHYLETHYL)-, (1S...","Cyclohexanol, 5-methyl-2-(1-methylethyl)-, [1S...",RefChem:791898,DTXCID701024626,MFCD00062984,"(1S,2R,5R)-5-methyl-2-propan-2-ylcyclohexan-1-ol","(1S,2R,5R)-2-Isopropyl-5-methylcyclohexanol","(1S,2R,5R)-5-methyl-2-(propan-2-yl)cyclohexan-..."
1,"(1R)-2,6,6-Trimethylbicyclo[3.1.1]hept-2-ene",82227,(+)-alpha-Pinene,7785-70-8,"(1R,5R)-2,6,6-Trimethylbicyclo[3.1.1]hept-2-ene",(1R)-(+)-alpha-Pinene,(+)-2-Pinene,"alpha-Pinene, (+)-",d-.alpha.-Pinene,alpha-Pinene(dextro),...,H6CM4TWH1W,"(1R,5R)-pin-2-ene","Bicyclo[3.1.1]hept-2-ene, 2,6,6-trimethyl-, (1...",CCRIS 9059,DTXSID7041671,EINECS 232-087-8,".ALPHA.-PINENE, D-","(+)-3,6,6-TRIMETHYLBICYCLO[3.1.1]HEPT-2-ENE",.ALPHA.-PINENE-(+),(1R)-.ALPHA.-PINENE
2,(E)-Hexadec-2-enal,5280541,2-Hexadecenal,3163-37-9,Z79H35Z0RU,2-(d5)-hexadecenal,RefChem:909856,(E)-2-hexadecenal,22644-96-8,(E)-hexadec-2-enal,...,trans-2-Hexadecenal,(2E)-2-Hexadecenal,(2E)-hexadec-2-enal,2-trans-Hexadecenal,trans-hexadec-2-enal,27104-14-9,UNII-Z79H35Z0RU,"2-hexadecenal, E",16:1 aldehyde,"2-Hexadecenal, (E)-"
3,.alpha. Isomethyl ionone,5372174,"3-Methyl-4-(2,6,6-trimethyl-2-cyclohexen-1-yl)...","(3E)-3-methyl-4-(2,6,6-trimethylcyclohex-2-en-...",RefChem:1068240,204-846-3,Alpha-isomethylionone,127-51-5,Cetone alpha,alpha-iso-Methylionone,...,"(E)-3-methyl-4-(2,6,6-trimethylcyclohex-2-en-1...",FEMA No. 2714,"Isomethylionone, alpha-",.alpha.-Cetone,a-iso-Methylionone,"3-BUTEN-2-ONE, 3-METHYL-4-(2,6,6-TRIMETHYL-2-C...",NSC 66432,.alpha. Isomethyl ionone,Isomethyl-.alpha.-ionone,15789-90-9
4,.alpha.-Methyl-.alpha.-[4-methyl-3-pentenyl]ox...,102611,Linalool oxide (5-ring),RefChem:799198,6-methyl-2-(oxiran-2-yl)hept-5-en-2-ol,Epoxydihydrolinalool,UNII-4UJJ55KMCS,EINECS 215-723-9,SCHEMBL891431,"3,7-DIMETHYL-1,2-EPOXY-3-HYDROXY-6-OCTENE",...,NS00012788,6-Methyl-2-(2-oxiranyl)-5-hepten-2-ol #,alpha-methyl-alpha-[4-methyl-3-pentenyl]oriran...,alpha-methyl-alpha-[4-methyl-3-pentenyl]oxiran...,alpha-Methyl-alpha-[4-methyl-3-pentenyl]oxiran...,.alpha.-Methyl-.alpha.-[4-methyl-3-pentenyl]ox...,NaN,NaN,NaN,NaN


## NIST Chemistry WebBook

In [ ]:
# for index, row in df.iterrows():
#     inchi = row["InChI"]
#     if pd.isna(inchi):
#         print(f"No InChI for {row['Input Name']}, skipping NIST lookup")
#         continue

nist_compound = nist.get_compound(df["InChI"][0])
print(nist_compound.ID)

nist_compound.get_gas_chromatography()

gc_all = nist_compound.gas_chromat

for gc in gc_all:
    print(gc)
    print(gc.data)
    # print(gc.compound, gc.ri_type, gc.column_type, gc.temp_regime)
    print(gc.data.columns)
    # print(gc.data[['I']])
    # gc.data

# get I where ri_type = "Normal alkane RI" and column_type = "non-polar"

C490993
Chromatogram(C490993, Kovats' RI, non-polar column, temperature ramp: 3 data points)
  Column type Active phase Column length (m) Carrier gas Substrate  \
0   Capillary         BP-1               25.          N2             
1   Capillary         BP-1               25.                         
2   Capillary         HP-5               25.          H2             

  Column diameter (mm) Phase thickness (μm) Tstart (C) Tend (C)  \
0                  0.5                 0.25        60.     220.   
1                  0.5                 0.25        60.     220.   
2                 0.32                 0.52        40.     240.   

  Heat rate (K/min) Initial hold (min) Final hold (min)      I  \
0                5.                                 10.  1174.   
1                5.                                 10.  1174.   
2                4.                                      1182.   

                                           Reference  Comment  
0  Rajeswara Rao, B.R., Biom